# Phutball Elo: round robin, the browser opponent as a player, and your games

1. **Round robin** between the self-play checkpoints close enough to beat each other (it50 = 1000 reference, it80, it90,
   it100 = final, the 17x13 final on 21x15, the best PBT member): 32x16 Gumbel search without root noise, temperature
   0.25 sampling, 64 games per colour per pair, 360-turn cap. Each match line reports how many games were distinct.
2. **The browser opponent as its own player** ("it100-puct400": the final network with the browser's PUCT search at
   400 playouts) plays every round-robin player, one process per opponent, 10 games per colour each, from the empty board with the Gumbel side sampling at 0.25 (the round robin's conditions).
3. **Refit** with both, plus your games against the browser opponent in HUMAN. Output: `phutball/elo/elo.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/phutball'

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install mctx==0.0.6 2>&1 | tail -1

In [ ]:
import os, subprocess, sys
S = f'{DRIVE}/selfplay'; E = f'{DRIVE}/elo'; os.makedirs(E, exist_ok=True)
TEMP = 0.25       # move sampling in BOTH the round robin and the browser-bot games
OPEN_MOVES = 0    # 0 = sample the whole game; if distinct-games counts are low, prefer TEMP = 1.0 with OPEN_MOVES = 20
                  # (sample only the opening, greedy after: varied games, full-strength tactics) over raising TEMP everywhere
PLAYERS = {'it50': f'{S}/selfplay_21x15_switch_s128_it50.pkl', 'it80': f'{S}/selfplay_21x15_best_it80.pkl',
           'it90': f'{S}/selfplay_21x15_best_it90.pkl', 'it100': f'{S}/selfplay_21x15_best_it100.pkl',
           '17x13': f'{S}/selfplay_17x13_final.pkl', 'pbt': f'{DRIVE}/grpo2/best_latest.pkl'}
PLAYERS = {k: v for k, v in PLAYERS.items() if os.path.exists(v) or print('missing, skipped:', k, v)}
def run(cmd, log=None, env=None):
    p = subprocess.Popen(cmd, shell=True, cwd='/content/phutball-jax', env=env, text=True,
                         stdout=open(log, 'w') if log else subprocess.PIPE, stderr=subprocess.STDOUT)
    if not log:
        for line in p.stdout: print(line, end='')
    return p
# 1. round robin
players = ' '.join(f'--player {k}={v}' for k, v in PLAYERS.items())
print('exit', run(f'{sys.executable} -u -m expert.elo_tournament {players} --anchor it50 --games 64 --sims 32 --temperature {TEMP} --open-moves {OPEN_MOVES} --out {E}/elo_rr.json').wait())

In [ ]:
# 2. the browser opponent (final + PUCT-400) against every player, in parallel
env = dict(os.environ, XLA_PYTHON_CLIENT_PREALLOCATE='false')
procs = {k: run(f'{sys.executable} -u -m expert.puct_calibration --params {PLAYERS["it100"]} --playouts 400 --gumbel-sims 32 '
                f'--games 10 --temperature {TEMP} --open-moves {OPEN_MOVES} --opponent {k}={v} --out {E}/puct_vs_{k}.json', log=f'{E}/puct_vs_{k}.log', env=env)
         for k, v in PLAYERS.items()}
for k, p in procs.items():
    print(k, 'exit', p.wait(), '|', open(f'{E}/puct_vs_{k}.log').read().strip().splitlines()[-1])

In [ ]:
# 3. refit: round robin + browser-opponent games + your games against the browser opponent
HUMAN = []   # e.g. ["you:3-7-0 vs it100-puct400"]  (your games are against the browser = final + PUCT-400)
extra = ' '.join(f'--extra-results {E}/puct_vs_{k}.json' for k in PLAYERS if os.path.exists(f'{E}/puct_vs_{k}.json'))
human = ' '.join(f'--human "{h}"' for h in HUMAN)
print('exit', run(f'{sys.executable} -u -m expert.elo_tournament --player it50={PLAYERS["it50"]} --anchor it50 '
                  f'--from-results {E}/elo_rr.json {extra} {human} --out {E}/elo.json').wait())